# MXene Literature-to-ML Electrode Descriptor Extractor

Upload one paper or process a folder of PDFs. The notebook produces two linked tables:

1. **Electrode table:** one row per physically distinct material/sample.
2. **Measurement table:** one row per explicitly reported electrochemical observation, with the electrode descriptors copied into that row for ML.

This prevents two common errors:

- treating scan-rate or current-density sweeps as different materials;
- retaining only the best capacitance and discarding the conditions that explain it.

The schema combines descriptors used in literature-based capacitance models with MXene-specific variables. It covers composition, synthesis, architecture, pore structure, surface chemistry, electrode fabrication, electrolyte/cell conditions, capacitance, capacity, energy/power density, cycling, and EIS.

**Literature anchors used to define the schema**

- Su et al., *Nanoscale Advances* (2019), DOI: 10.1039/C9NA00105K
- Saad et al., *Journal of Energy Storage* (2022), DOI: 10.1016/j.est.2022.105411
- Mishra et al., *Scientific Reports* (2023), DOI: 10.1038/s41598-023-33524-1
- Krishna and Mir, *Energy Advances* (2024), DOI: 10.1039/D4YA00460D
- Kawai et al., *Small Methods* (2025), DOI: 10.1002/smtd.202400062
- Vasistha et al., *Journal of Energy Storage* (2026), DOI: 10.1016/j.est.2026.121921

The extractor is intentionally **sparse**: it returns only fields supported by the paper. Missing columns are added as null during flattening. Each extracted value retains its original unit, source location, short evidence text, and confidence flag.

## Setup

In [1]:
%pip install --quiet openai pypdf pandas
print("done — restart the kernel if this was the first install")

Note: you may need to restart the kernel to use updated packages.
done — restart the kernel if this was the first install



[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# Or hardcode (don't commit the file if you do):
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

key loaded: True


In [2]:
import json, re, time, pathlib, traceback
import pandas as pd
from pypdf import PdfReader
from openai import OpenAI

# All generated files are written under this parent folder.
OUTPUT_DIR = pathlib.Path("V2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI()

# gpt-5.4        $2.50 / $15.00 per 1M tok  -- recommended
# gpt-5.6-terra  $2.50 / $15.00             -- newer family, comparable
# gpt-5.6-luna   $1.00 /  $6.00             -- ~60% cheaper, good for bulk
# gpt-5.4-mini   $0.75 /  $4.50             -- budget; validate before trusting
MODEL = "gpt-5.6-terra"

PRICING = {
    "gpt-5.4":       (2.50, 15.00),
    "gpt-5.6-terra": (2.50, 15.00),
    "gpt-5.6-sol":   (5.00, 30.00),
    "gpt-5.6-luna":  (1.00,  6.00),
    "gpt-5.4-mini":  (0.75,  4.50),
    "gpt-5.4-nano":  (0.20,  1.25),
}
print("model:", MODEL)

model: gpt-5.6-terra


## Record granularity: electrode versus measurement

A clean literature dataset needs two identifiers.

### A separate electrode/sample
Create a new electrode when the physical material changes, for example:

- MXene formula or parent MAX phase;
- etchant, synthesis route, intercalant, delamination, annealing, or surface treatment;
- composite partner or composition ratio;
- film architecture, loading formulation, or a fabricated control/reference sample.

### A separate measurement
Keep the same electrode but create a new measurement when only the test condition or reported output changes, for example:

- scan rate or current density;
- two-electrode versus three-electrode testing;
- electrolyte, concentration, potential window, or temperature;
- gravimetric, areal, or volumetric capacitance;
- cycling retention, rate retention, energy/power density, or EIS result.

Do not extract benchmark values cited from other papers. Do not digitize curves automatically. Extract only exact values printed in the text, tables, or figure captions. Repeated mentions of the same result in the abstract, results, and conclusion should become one measurement, not duplicates.

`MEASUREMENT_MODE = "all_explicit"` is recommended for ML. It retains every exact, explicitly reported performance point. Use `"representative"` when cost or output size is more important than preserving sweeps.

In [3]:
# ---- Schema version and extraction mode ---------------------------------
SCHEMA_VERSION = "2.0-literature-ml"
MEASUREMENT_MODE = "all_explicit"   # "all_explicit" or "representative"


def F(category, tier, description, preferred_unit=None, literature_basis=""):
    return {
        "category": category,
        "tier": tier,
        "description": description,
        "preferred_unit": preferred_unit,
        "literature_basis": literature_basis,
    }


# Paper metadata are direct scalar values. Electrode and measurement fields are
# sparse arrays with value-level provenance.
PAPER_FIELDS = {
    "doi": "Digital object identifier",
    "title": "Full article title",
    "journal": "Journal or conference name",
    "year": "Publication year as YYYY",
    "authors": "Full author list when available",
}


# One record per physically distinct electrode/sample.
ELECTRODE_FEATURES = {
    # Identity and composition
    "mxene_formula": F("identity", "core", "Normalized MXene formula, e.g. Ti3C2Tx or Mo2TiC2Tx", None, "Krishna 2024; Kawai 2025"),
    "parent_max_phase": F("identity", "extended", "Parent MAX-phase formula used to synthesize the MXene", None, "MXene literature"),
    "composition": F("identity", "core", "Complete active-material composition including MXene and all partners", None, "Mishra 2023; Saad 2022"),
    "mxene_family": F("identity", "extended", "Carbide, nitride, or carbonitride and M_n+1X_n family", None, "MXene literature"),
    "composite_partner": F("identity", "core", "Non-MXene active component or support, e.g. rGO, CNT, MnO2, polymer", None, "Saad 2022; Vasistha 2026"),
    "composite_ratio": F("identity", "core", "Reported mass, molar, or volume ratio among active components", None, "Saad 2022"),
    "mxene_weight_fraction": F("identity", "core", "MXene content in the active electrode or composite", "wt%", "Krishna 2024"),
    "dopant_elements": F("identity", "extended", "Intentionally introduced dopant element or elements", None, "Su 2019; Mishra 2023"),
    "dopant_content": F("identity", "extended", "Reported dopant loading or concentration", "at% or wt%", "Su 2019; Mishra 2023"),
    "intercalant": F("identity", "core", "Molecule, ion, polymer, or species intentionally inserted between layers", None, "Kawai 2025; MXene literature"),
    "preintercalated_ion": F("identity", "core", "Ion present before electrochemical testing as a pillar or pre-intercalant", None, "Kawai 2025"),
    "surface_termination_summary": F("surface chemistry", "core", "Reported surface terminations or functional groups, such as -O, -OH, -F, -Cl", None, "Kawai 2025; MXene literature"),
    "electrode_architecture": F("identity", "core", "Film, paper, hydrogel, aerogel, foam, fiber, printed structure, coating, or powder electrode", None, "Mishra 2023"),
    "morphology": F("identity", "core", "Reported morphology, e.g. accordion-like, nanosheet, porous network, aligned lamellae", None, "Mishra 2023; Saad 2022"),
    "layer_state": F("identity", "core", "Single-layer, few-layer, multilayer, delaminated, or restacked state", None, "MXene literature"),

    # Synthesis and processing
    "synthesis_method": F("synthesis", "core", "Overall route from precursor to final active material", None, "Vasistha 2026; MXene literature"),
    "etchant": F("synthesis", "core", "Etching chemistry, e.g. HF, LiF/HCl, NH4HF2, molten salt, alkali", None, "MXene literature"),
    "etchant_concentration": F("synthesis", "extended", "Etchant concentration or component amounts", "M, wt%, or reported", "MXene literature"),
    "etching_temperature": F("synthesis", "extended", "Temperature used during MAX-phase etching", "degC", "MXene literature"),
    "etching_time": F("synthesis", "extended", "Duration of etching", "h", "MXene literature"),
    "delamination_method": F("synthesis", "core", "Method used to delaminate or exfoliate multilayer MXene", None, "MXene literature"),
    "delamination_agent": F("synthesis", "extended", "Chemical or ion used to assist delamination", None, "MXene literature"),
    "sonication_time": F("synthesis", "extended", "Sonication duration associated with delamination or dispersion", "min or h", "MXene literature"),
    "washing_endpoint_ph": F("synthesis", "extended", "Final pH or washing criterion after etching", None, "MXene literature"),
    "post_treatment": F("synthesis", "core", "Chemical, thermal, electrochemical, plasma, freeze-drying, templating, or other post-treatment", None, "Vasistha 2026"),
    "annealing_temperature": F("synthesis", "extended", "Post-synthesis annealing temperature", "degC", "MXene literature"),
    "annealing_time": F("synthesis", "extended", "Post-synthesis annealing duration", "h", "MXene literature"),
    "composite_fabrication_method": F("synthesis", "extended", "Mixing, in-situ growth, self-assembly, filtration, printing, coating, or other composite route", None, "Saad 2022"),
    "drying_method": F("synthesis", "extended", "Vacuum, ambient, freeze, supercritical, or other drying method", None, "MXene literature"),
    "drying_temperature": F("synthesis", "extended", "Drying temperature for material or fabricated electrode", "degC", "MXene literature"),

    # Geometric, structural, and transport descriptors
    "interlayer_spacing": F("structure", "core", "Basal-plane d-spacing, normally derived from the XRD (002) reflection", "angstrom or nm", "Kawai 2025"),
    "xrd_002_peak": F("structure", "extended", "Position of the MXene (002) diffraction peak", "degree 2theta", "Kawai 2025"),
    "flake_size": F("structure", "core", "Lateral sheet or flake dimension", "nm or um", "MXene literature"),
    "sheet_thickness": F("structure", "core", "Thickness of an individual MXene sheet or stack when explicitly reported", "nm", "Krishna 2024"),
    "electrode_thickness": F("structure", "core", "Thickness of the fabricated film or electrode", "um or mm", "Krishna 2024"),
    "mass_loading": F("structure", "core", "Active-material loading normalized by geometric area", "mg cm-2", "Mishra 2023; Vasistha 2026"),
    "electrode_density": F("structure", "extended", "Packing, film, or apparent electrode density", "g cm-3", "MXene literature"),
    "specific_surface_area": F("structure", "core", "BET or otherwise reported specific surface area", "m2 g-1", "Su 2019; Saad 2022; Mishra 2023; Krishna 2024"),
    "total_pore_volume": F("structure", "core", "Total pore volume", "cm3 g-1", "Su 2019; Saad 2022; Mishra 2023"),
    "micropore_volume": F("structure", "extended", "Micropore volume", "cm3 g-1", "Porous-carbon ML literature"),
    "mesopore_volume": F("structure", "extended", "Mesopore volume", "cm3 g-1", "Porous-carbon ML literature"),
    "pore_diameter": F("structure", "core", "Average, modal, or median pore diameter with statistic retained in evidence", "nm", "Su 2019; Saad 2022; Mishra 2023"),
    "porosity": F("structure", "extended", "Total open or apparent porosity", "%", "MXene literature"),
    "tortuosity": F("structure", "extended", "Reported ion-transport tortuosity", None, "Electrode transport literature"),
    "raman_id_ig": F("structure", "core", "Raman D-to-G intensity ratio for carbon-containing electrodes", None, "Su 2019; Saad 2022; Mishra 2023"),
    "electrical_conductivity": F("transport", "core", "Electronic conductivity of the material, film, or electrode", "S cm-1 or S m-1", "MXene literature"),
    "sheet_resistance": F("transport", "extended", "Electrical sheet resistance", "ohm sq-1", "MXene film literature"),
    "contact_angle": F("transport", "extended", "Water or electrolyte contact angle", "degree", "Surface-wetting literature"),

    # Elemental and termination chemistry
    "carbon_atomic_pct": F("surface chemistry", "core", "Carbon atomic percentage from XPS or elemental analysis", "at%", "Saad 2022"),
    "nitrogen_atomic_pct": F("surface chemistry", "core", "Nitrogen atomic percentage", "at%", "Su 2019; Saad 2022; Mishra 2023"),
    "oxygen_atomic_pct": F("surface chemistry", "core", "Oxygen atomic percentage", "at%", "Su 2019; Saad 2022; Mishra 2023"),
    "fluorine_atomic_pct": F("surface chemistry", "core", "Fluorine atomic percentage", "at%", "MXene termination literature"),
    "sulfur_atomic_pct": F("surface chemistry", "extended", "Sulfur atomic percentage", "at%", "Heteroatom-doping literature"),
    "other_heteroatom_atomic_pct": F("surface chemistry", "extended", "Other reported heteroatom percentage; preserve element in value", "at%", "Heteroatom-doping literature"),
    "o_termination_pct": F("surface chemistry", "extended", "Quantified =O or O termination fraction", "% or at%", "MXene termination literature"),
    "oh_termination_pct": F("surface chemistry", "extended", "Quantified -OH termination fraction", "% or at%", "MXene termination literature"),
    "f_termination_pct": F("surface chemistry", "extended", "Quantified -F termination fraction", "% or at%", "MXene termination literature"),
    "oxidation_or_tio2_fraction": F("surface chemistry", "extended", "Reported oxidation level or TiO2 fraction in the electrode", "% or ratio", "MXene oxidation literature"),

    # Electrode formulation and fabrication
    "freestanding": F("fabrication", "core", "Whether the tested electrode is binder-free and self-supporting", "yes/no", "MXene electrode literature"),
    "binder": F("fabrication", "core", "Binder identity", None, "Electrode formulation literature"),
    "binder_fraction": F("fabrication", "extended", "Binder fraction in the electrode formulation", "wt%", "Electrode formulation literature"),
    "conductive_additive": F("fabrication", "core", "Conductive additive identity", None, "Electrode formulation literature"),
    "conductive_additive_fraction": F("fabrication", "extended", "Conductive additive fraction", "wt%", "Electrode formulation literature"),
    "active_material_fraction": F("fabrication", "extended", "Active-material fraction in slurry or electrode", "wt%", "Electrode formulation literature"),
    "current_collector": F("fabrication", "core", "Current collector material", None, "Electrochemical methods"),
    "substrate": F("fabrication", "extended", "Supporting substrate if different from current collector", None, "Electrochemical methods"),
    "electrode_area": F("fabrication", "extended", "Geometric tested area", "cm2", "Electrochemical methods"),
    "pressing_pressure": F("fabrication", "extended", "Pressure used to compact or laminate the electrode", "MPa", "Electrode fabrication literature"),
}


# One record per exact electrochemical observation associated with an electrode.
MEASUREMENT_FEATURES = {
    # Test and cell conditions
    "test_type": F("test protocol", "core", "GCD, CV, EIS, cycling, rate capability, or other test", None, "All capacitance ML studies"),
    "capacitance_calculation_method": F("test protocol", "core", "Method used to calculate capacitance, including equation basis if stated", None, "Mishra 2023"),
    "cell_configuration": F("cell", "core", "Two-electrode or three-electrode configuration", None, "Saad 2022; Mishra 2023; Krishna 2024"),
    "device_configuration": F("cell", "core", "Symmetric, asymmetric, hybrid, micro-supercapacitor, or single working electrode", None, "Mishra 2023"),
    "electrode_role_in_cell": F("cell", "extended", "Positive, negative, both, or working-electrode role", None, "Electrochemical methods"),
    "electrolyte": F("electrolyte", "core", "Full electrolyte identity as reported", None, "Mishra 2023; Krishna 2024; Vasistha 2026"),
    "electrolyte_concentration": F("electrolyte", "core", "Electrolyte concentration", "M, m, wt%, or reported", "Saad 2022; Krishna 2024; Vasistha 2026"),
    "solvent": F("electrolyte", "extended", "Electrolyte solvent or solvent mixture", None, "Electrolyte literature"),
    "cation": F("electrolyte", "core", "Primary mobile cation", None, "Krishna 2024; Kawai 2025"),
    "anion": F("electrolyte", "core", "Primary mobile anion", None, "Krishna 2024"),
    "cation_valence": F("electrolyte", "extended", "Formal charge of the mobile cation", None, "Electrolyte descriptor engineering"),
    "cation_ionic_radius": F("electrolyte", "extended", "Ionic radius explicitly reported or supplied by the paper", "pm or angstrom", "Electrolyte descriptor engineering"),
    "cation_mobility": F("electrolyte", "core", "Cation mobility used as an ML descriptor", "cm2 V-1 s-1", "Krishna 2024"),
    "anion_mobility": F("electrolyte", "core", "Anion mobility used as an ML descriptor", "cm2 V-1 s-1", "Krishna 2024"),
    "electrolyte_ionic_conductivity": F("electrolyte", "core", "Ionic conductivity of the electrolyte", "S cm-1", "Saad 2022"),
    "electrolyte_ph": F("electrolyte", "extended", "Electrolyte pH", None, "Electrochemical methods"),
    "counter_electrode": F("cell", "extended", "Counter-electrode identity", None, "Electrochemical methods"),
    "reference_electrode": F("cell", "extended", "Reference-electrode identity", None, "Electrochemical methods"),
    "separator": F("cell", "extended", "Separator or membrane identity", None, "Device methods"),
    "test_temperature": F("test protocol", "extended", "Electrochemical test temperature", "degC or K", "Testing-protocol literature"),
    "potential_lower": F("test protocol", "core", "Lower potential or voltage bound", "V", "Mishra 2023"),
    "potential_upper": F("test protocol", "core", "Upper potential or voltage bound", "V", "Mishra 2023"),
    "potential_window": F("test protocol", "core", "Absolute operating potential or voltage window", "V", "Su 2019; Saad 2022; Mishra 2023; Krishna 2024"),
    "scan_rate": F("test protocol", "core", "CV scan rate associated with this observation", "mV s-1", "Krishna 2024"),
    "current_density": F("test protocol", "core", "GCD current density associated with this observation", "A g-1 or mA cm-2", "Saad 2022; Mishra 2023; Krishna 2024; Vasistha 2026"),
    "cycle_number": F("test protocol", "extended", "Cycle index at which this observation was recorded", None, "Cycling literature"),

    # Prediction targets and related electrochemical outputs
    "gravimetric_capacitance": F("target", "core", "Specific or gravimetric capacitance", "F g-1", "All capacitance ML studies"),
    "areal_capacitance": F("target", "core", "Areal capacitance", "F cm-2 or mF cm-2", "Supercapacitor literature"),
    "volumetric_capacitance": F("target", "core", "Volumetric capacitance", "F cm-3", "MXene literature"),
    "specific_capacity": F("target", "extended", "Specific capacity when the material is reported in battery-type units", "C g-1 or mAh g-1", "Hybrid-electrode literature"),
    "energy_density": F("target", "core", "Reported energy density with normalization basis retained", "Wh kg-1, Wh L-1, or uWh cm-2", "Soni 2025"),
    "power_density": F("target", "core", "Reported power density with normalization basis retained", "W kg-1, W L-1, or mW cm-2", "Soni 2025"),
    "capacitance_retention": F("target", "core", "Capacitance retained after cycling", "%", "Soni 2025"),
    "rate_retention": F("target", "extended", "Capacitance or capacity retained at a higher rate relative to a stated baseline", "%", "Rate-capability literature"),
    "cycles_tested": F("target", "core", "Total cycle count associated with the reported retention", None, "Soni 2025"),
    "coulombic_efficiency": F("target", "extended", "Coulombic efficiency", "%", "Cycling literature"),
    "esr": F("impedance", "core", "Equivalent series resistance", "ohm", "Saad 2022"),
    "charge_transfer_resistance": F("impedance", "core", "Charge-transfer resistance Rct", "ohm", "Saad 2022"),
    "relaxation_time": F("impedance", "extended", "Characteristic relaxation time", "s", "EIS literature"),
    "diffusion_coefficient": F("impedance", "extended", "Reported ion diffusion coefficient", "cm2 s-1", "Transport literature"),
    "b_value": F("kinetics", "extended", "Power-law b value from log(i)-log(v) analysis", None, "CV kinetics literature"),
    "capacitive_contribution": F("kinetics", "extended", "Surface-controlled or capacitive contribution at the stated scan rate", "%", "CV kinetics literature"),
}

TARGET_FIELDS = {
    "gravimetric_capacitance", "areal_capacitance", "volumetric_capacitance",
    "specific_capacity", "energy_density", "power_density", "capacitance_retention",
    "rate_retention", "coulombic_efficiency", "esr", "charge_transfer_resistance",
    "relaxation_time", "diffusion_coefficient", "b_value", "capacitive_contribution",
}


def _sparse_field_array_schema(allowed_names, description):
    return {
        "type": "array",
        "description": description,
        "items": {
            "type": "object",
            "additionalProperties": False,
            "required": ["name", "value", "unit", "source_location", "evidence", "confidence"],
            "properties": {
                "name": {"type": "string", "enum": list(allowed_names)},
                "value": {"type": "string", "description": "Exact reported value or concise categorical text."},
                "unit": {"type": ["string", "null"], "description": "Unit exactly as reported; do not convert."},
                "source_location": {"type": "string", "description": "Section, table, figure, caption, or page location."},
                "evidence": {"type": "string", "description": "Short supporting excerpt, preferably no more than 25 words."},
                "confidence": {
                    "type": "string",
                    "enum": ["stated", "derived", "uncertain"],
                    "description": "stated=explicit for this record; derived=shared method or simple printed relation; uncertain=attribution ambiguous.",
                },
            },
        },
    }


SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["paper", "n_electrodes", "n_measurements", "electrodes", "summary"],
    "properties": {
        "paper": {
            "type": "object",
            "additionalProperties": False,
            "required": list(PAPER_FIELDS),
            "properties": {
                key: {"type": ["string", "null"], "description": desc}
                for key, desc in PAPER_FIELDS.items()
            },
        },
        "n_electrodes": {"type": "integer"},
        "n_measurements": {"type": "integer"},
        "electrodes": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["electrode_id", "label", "is_primary", "role", "fields", "n_measurements", "measurements"],
                "properties": {
                    "electrode_id": {"type": "string", "description": "Stable within-paper ID such as E01, E02."},
                    "label": {"type": "string", "description": "Authors' sample name, verbatim where possible."},
                    "is_primary": {"type": "boolean"},
                    "role": {
                        "type": "string",
                        "enum": ["primary", "variant", "control", "composite", "reference"],
                    },
                    "fields": _sparse_field_array_schema(
                        ELECTRODE_FEATURES,
                        "Only electrode/material fields supported by the paper. Omit unreported fields.",
                    ),
                    "n_measurements": {"type": "integer"},
                    "measurements": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "additionalProperties": False,
                            "required": ["measurement_id", "measurement_label", "is_representative", "fields"],
                            "properties": {
                                "measurement_id": {"type": "string", "description": "Stable ID such as E01-M01."},
                                "measurement_label": {"type": "string", "description": "Concise condition/result label."},
                                "is_representative": {"type": "boolean", "description": "True for the headline/baseline result for this electrode."},
                                "fields": _sparse_field_array_schema(
                                    MEASUREMENT_FEATURES,
                                    "Only condition and output fields supported for this exact observation.",
                                ),
                            },
                        },
                    },
                },
            },
        },
        "summary": {
            "type": "string",
            "description": "3-5 sentences covering the sample set, measurement coverage, headline result, and important missing descriptors.",
        },
    },
}


def feature_catalog_dataframe():
    rows = []
    for level, catalog in (("electrode", ELECTRODE_FEATURES), ("measurement", MEASUREMENT_FEATURES)):
        for name, meta in catalog.items():
            rows.append({"level": level, "feature": name, **meta})
    return pd.DataFrame(rows)


print(
    f"schema {SCHEMA_VERSION}: {len(PAPER_FIELDS)} paper fields, "
    f"{len(ELECTRODE_FEATURES)} electrode features, "
    f"{len(MEASUREMENT_FEATURES)} measurement features"
)
feature_catalog_dataframe().head(12)

schema 2.0-literature-ml: 5 paper fields, 68 electrode features, 42 measurement features


,level,feature,category,tier,description,preferred_unit,literature_basis
0,electrode,mxene_formula,identity,core,"Normalized MXene formula, e.g. Ti3C2Tx or Mo2T...",None,Krishna 2024; Kawai 2025
1,electrode,parent_max_phase,identity,extended,Parent MAX-phase formula used to synthesize th...,None,MXene literature
2,electrode,composition,identity,core,Complete active-material composition including...,None,Mishra 2023; Saad 2022
3,electrode,mxene_family,identity,extended,"Carbide, nitride, or carbonitride and M_n+1X_n...",None,MXene literature
4,electrode,composite_partner,identity,core,"Non-MXene active component or support, e.g. rG...",None,Saad 2022; Vasistha 2026
5,electrode,composite_ratio,identity,core,"Reported mass, molar, or volume ratio among ac...",None,Saad 2022
6,electrode,mxene_weight_fraction,identity,core,MXene content in the active electrode or compo...,wt%,Krishna 2024
7,electrode,dopant_elements,identity,extended,Intentionally introduced dopant element or ele...,None,Su 2019; Mishra 2023
8,electrode,dopant_content,identity,extended,Reported dopant loading or concentration,at% or wt%,Su 2019; Mishra 2023
9,electrode,intercalant,identity,core,"Molecule, ion, polymer, or species intentional...",None,Kawai 2025; MXene literature


## PDF text extraction

A note from testing this on real two-column ACS papers: the common advice to use
`pdftotext -layout` is wrong for this document class. Layout mode preserves horizontal
position, which interleaves the left and right columns line by line and shreds every
sentence. Reading-order extraction is what you want, and `pypdf` does it correctly.

In [4]:
def extract_pdf_text(path, verbose=True):
    """Extract full text in reading order. Returns (text, n_pages)."""
    reader = PdfReader(path)
    pages = [p.extract_text() or "" for p in reader.pages]
    text = "\n\n".join(pages)
    text = re.sub(r"[ \t]{3,}", " ", text)
    text = re.sub(r"\n{4,}", "\n\n", text)

    if verbose:
        chars = len(text)
        print(f"pages: {len(pages)}   chars: {chars:,}   est. tokens: ~{chars//4:,}")
        if chars / max(len(pages), 1) < 500:
            print("\n  WARNING: very little text per page — likely a scanned PDF.")
            print("  Run OCR first (ocrmypdf in.pdf out.pdf) before extracting.")
        print("\n--- first 400 chars ---")
        print(text[:400].strip())
    return text, len(pages)

print("ready")

ready


## Structured extraction call

The output is sparse to reduce hallucination and token cost. A field is included only when the paper explicitly reports it or when a shared methods statement clearly applies to the record. The flattening functions later create the complete fixed column set.

The extraction prompt performs three operations in order:

1. enumerate physical electrode samples;
2. extract material descriptors for each sample;
3. attach condition-specific electrochemical measurements to the correct sample.

The model must preserve exact units and include a source location and short evidence excerpt for every extracted value.

In [5]:
def _catalog_guide(catalog):
    lines = []
    current = None
    for name, meta in catalog.items():
        if meta["category"] != current:
            current = meta["category"]
            lines.append(f"\n[{current.upper()}]")
        unit = f" Preferred unit: {meta['preferred_unit']}." if meta["preferred_unit"] else ""
        lines.append(f"- {name}: {meta['description']}.{unit}")
    return "\n".join(lines)


ELECTRODE_GUIDE = _catalog_guide(ELECTRODE_FEATURES)
MEASUREMENT_GUIDE = _catalog_guide(MEASUREMENT_FEATURES)

SYSTEM_PROMPT_TEMPLATE = """You extract auditable electrode and electrochemical data from MXene and related supercapacitor research papers.

DATA MODEL
- An ELECTRODE is a physically distinct material/sample.
- A MEASUREMENT is one exact electrochemical observation associated with that electrode.
- Different current densities, scan rates, electrolytes, cell configurations, cycle counts, or output bases are measurements, not new electrodes.

ELECTRODE SPLITTING
Create separate electrodes for different formulae, synthesis routes, etchants, intercalants, treatments, annealing conditions, composite partners or ratios, architectures, and fabricated controls/references.
Do not create electrodes from literature-comparison values belonging to other papers.
Exactly one electrode should have is_primary=true.

MEASUREMENT MODE: {measurement_mode}
- all_explicit: retain every distinct numerical performance point explicitly printed in text, a table, or a caption. A scan/current sweep may therefore produce several measurements for one electrode.
- representative: retain the headline or lowest-rate baseline capacitance for each basis/configuration, plus reported rate, cycling, energy/power, and EIS summaries.
Do not digitize plots. Do not create duplicate measurements when the same result is repeated in several sections.

EXTRACTION RULES
1. Read the entire paper before deciding the sample list.
2. Use authors' sample labels verbatim where possible and assign E01, E02... IDs.
3. Assign E01-M01, E01-M02... measurement IDs within each electrode.
4. Sparse output: include only fields supported by the paper. Never add a null field object.
5. Preserve the exact printed value and unit. Do not normalize or silently convert.
6. For each value, provide its section/table/figure location and a short supporting excerpt.
7. confidence=stated only when explicit for that exact sample/measurement.
8. confidence=derived when a shared methods condition clearly applies to several samples or when the paper itself supplies a direct calculation relation.
9. confidence=uncertain when sample attribution or condition matching is ambiguous.
10. Do not calculate capacitance from curves, infer missing chemistry, or supply external physical constants.
11. n_electrodes and all n_measurements counts must exactly match their arrays.

ELECTRODE FEATURE CATALOG
{electrode_guide}

MEASUREMENT FEATURE CATALOG
{measurement_guide}
"""


def build_system_prompt(measurement_mode=MEASUREMENT_MODE):
    if measurement_mode not in {"all_explicit", "representative"}:
        raise ValueError("measurement_mode must be 'all_explicit' or 'representative'")
    return SYSTEM_PROMPT_TEMPLATE.format(
        measurement_mode=measurement_mode,
        electrode_guide=ELECTRODE_GUIDE,
        measurement_guide=MEASUREMENT_GUIDE,
    )


def extract_descriptors(text, model=MODEL, max_chars=140_000, measurement_mode=MEASUREMENT_MODE):
    """One structured-output call. Returns (parsed_dict, usage_dict)."""
    if len(text) > max_chars:
        print(f"  note: truncating {len(text):,} -> {max_chars:,} chars")
        text = text[:max_chars]

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": build_system_prompt(measurement_mode)},
            {
                "role": "user",
                "content": "Extract all distinct electrodes and their linked measurements from this paper.\n\n" + text,
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "mxene_literature_ml", "schema": SCHEMA, "strict": True},
        },
    )
    parsed = json.loads(resp.choices[0].message.content)
    u = resp.usage
    return parsed, {"in": u.prompt_tokens, "out": u.completion_tokens, "model": model}


def report_cost(usage, n_elec=None, n_meas=None):
    rin, rout = PRICING.get(usage["model"], (2.50, 15.00))
    cost = usage["in"] / 1e6 * rin + usage["out"] / 1e6 * rout
    details = []
    if n_elec is not None:
        details.append(f"{n_elec} electrodes")
    if n_meas is not None:
        details.append(f"{n_meas} measurements")
    extra = f"   ({', '.join(details)})" if details else ""
    print(
        f"tokens  in={usage['in']:,}  out={usage['out']:,}   "
        f"cost ~${cost:.4f}   [{usage['model']}]{extra}"
    )
    return cost


print("ready")

ready


## Consistency and leakage checks

Structured output controls shape, not attribution quality. These checks flag missing IDs, duplicate fields, mismatched counts, measurement records without a target, and likely over-splitting. They also make paper- and electrode-level grouping available for later cross-validation.

In [6]:
def _field_map(fields):
    """Map a sparse field list by name. Later duplicates overwrite, but validation flags them."""
    return {item["name"]: item for item in fields}


def _duplicate_names(fields):
    names = [item.get("name") for item in fields]
    return sorted({name for name in names if name and names.count(name) > 1})


def validate(parsed, verbose=True):
    """Return warning strings. Empty means the structural checks passed."""
    warns = []
    electrodes = parsed.get("electrodes", [])

    if parsed.get("n_electrodes") != len(electrodes):
        warns.append(f"n_electrodes={parsed.get('n_electrodes')} but array has {len(electrodes)}")

    total_measurements = sum(len(e.get("measurements", [])) for e in electrodes)
    if parsed.get("n_measurements") != total_measurements:
        warns.append(
            f"n_measurements={parsed.get('n_measurements')} but nested arrays have {total_measurements}"
        )

    if not electrodes:
        warns.append("no electrodes extracted")
        return warns

    ids = [e.get("electrode_id", "").strip().lower() for e in electrodes]
    labels = [e.get("label", "").strip().lower() for e in electrodes]
    for kind, values in (("electrode IDs", ids), ("electrode labels", labels)):
        dupes = {v for v in values if v and values.count(v) > 1}
        if dupes:
            warns.append(f"duplicate {kind}: {sorted(dupes)}")

    n_primary = sum(1 for e in electrodes if e.get("is_primary"))
    if n_primary != 1:
        warns.append(f"{n_primary} electrodes marked primary; expected exactly 1")

    seen_measurement_ids = []
    for electrode in electrodes:
        e_label = electrode.get("label", electrode.get("electrode_id", "unknown"))
        fields = electrode.get("fields", [])
        duplicate_fields = _duplicate_names(fields)
        if duplicate_fields:
            warns.append(f"'{e_label}' duplicate electrode fields: {duplicate_fields}")

        invalid = [x.get("name") for x in fields if x.get("name") not in ELECTRODE_FEATURES]
        if invalid:
            warns.append(f"'{e_label}' invalid electrode fields: {invalid}")

        n_measurements = len(electrode.get("measurements", []))
        if electrode.get("n_measurements") != n_measurements:
            warns.append(
                f"'{e_label}' n_measurements={electrode.get('n_measurements')} but array has {n_measurements}"
            )
        if not fields:
            warns.append(f"'{e_label}' has no material descriptors")
        if n_measurements == 0:
            warns.append(f"'{e_label}' has no linked measurements")

        for measurement in electrode.get("measurements", []):
            mid = measurement.get("measurement_id", "").strip()
            if mid:
                seen_measurement_ids.append(mid.lower())
            m_fields = measurement.get("fields", [])
            duplicates = _duplicate_names(m_fields)
            if duplicates:
                warns.append(f"'{mid or e_label}' duplicate measurement fields: {duplicates}")
            invalid_m = [x.get("name") for x in m_fields if x.get("name") not in MEASUREMENT_FEATURES]
            if invalid_m:
                warns.append(f"'{mid or e_label}' invalid measurement fields: {invalid_m}")

            fmap = _field_map(m_fields)
            if not any(name in fmap for name in TARGET_FIELDS):
                warns.append(f"'{mid or e_label}' has no target or related electrochemical output")
            if not any(name in fmap for name in ("current_density", "scan_rate", "cycle_number", "test_type")):
                warns.append(f"'{mid or e_label}' has no test-rate/type descriptor")

    duplicate_measurements = {
        mid for mid in seen_measurement_ids if seen_measurement_ids.count(mid) > 1
    }
    if duplicate_measurements:
        warns.append(f"duplicate measurement IDs: {sorted(duplicate_measurements)}")

    if total_measurements > 100:
        warns.append(
            f"{total_measurements} measurements in one paper; inspect for duplicated narrative values or over-extraction"
        )

    if verbose:
        if warns:
            print("CHECKS — review these:")
            for warning in warns:
                print("  ! " + warning)
        else:
            print("checks passed")
    return warns


print("ready")

ready


## Flattening and review tables

The nested JSON is the audit record. The CSV tables are convenient views:

- `electrode_dataframe`: one row per material/sample;
- `measurement_dataframe`: one row per electrochemical observation, with electrode descriptors copied into the same row;
- `feature_catalog_dataframe`: the schema dictionary, categories, tiers, preferred units, and literature basis.

Use `provenance="full"` to add source and evidence columns, `"units"` for units and confidence only, or `"none"` for values only.

In [7]:
def _flatten_sparse_fields(fields, catalog, provenance="units"):
    fmap = _field_map(fields)
    row = {}
    for name in catalog:
        item = fmap.get(name)
        row[name] = item.get("value") if item else None
        if provenance in {"units", "full"}:
            row[name + "__unit"] = item.get("unit") if item else None
            row[name + "__conf"] = item.get("confidence") if item else None
        if provenance == "full":
            row[name + "__source"] = item.get("source_location") if item else None
            row[name + "__evidence"] = item.get("evidence") if item else None
    return row


def electrode_dataframe(parsed, source_file="", source_path="", provenance="units"):
    """One row per physically distinct electrode/sample."""
    paper = parsed.get("paper", {})
    rows = []
    for electrode in parsed.get("electrodes", []):
        rows.append({
            "schema_version": SCHEMA_VERSION,
            "source_file": source_file,
            "source_path": source_path,
            **paper,
            "electrode_id": electrode["electrode_id"],
            "electrode_label": electrode["label"],
            "role": electrode["role"],
            "is_primary": electrode["is_primary"],
            "n_measurements": len(electrode.get("measurements", [])),
            **_flatten_sparse_fields(electrode.get("fields", []), ELECTRODE_FEATURES, provenance),
        })
    return pd.DataFrame(rows)


def measurement_dataframe(parsed, source_file="", source_path="", provenance="units"):
    """One row per explicit electrochemical observation; ML-ready after unit normalization."""
    paper = parsed.get("paper", {})
    rows = []
    for electrode in parsed.get("electrodes", []):
        electrode_values = _flatten_sparse_fields(
            electrode.get("fields", []), ELECTRODE_FEATURES, provenance
        )
        for measurement in electrode.get("measurements", []):
            rows.append({
                "schema_version": SCHEMA_VERSION,
                "source_file": source_file,
                "source_path": source_path,
                **paper,
                "electrode_id": electrode["electrode_id"],
                "electrode_label": electrode["label"],
                "role": electrode["role"],
                "is_primary": electrode["is_primary"],
                "measurement_id": measurement["measurement_id"],
                "measurement_label": measurement["measurement_label"],
                "is_representative": measurement["is_representative"],
                **electrode_values,
                **_flatten_sparse_fields(
                    measurement.get("fields", []), MEASUREMENT_FEATURES, provenance
                ),
            })
    return pd.DataFrame(rows)


# Backward-compatible alias: the main corpus is now measurement-level, not electrode-level.
def to_long_dataframe(parsed, source_file="", source_path="", provenance="units"):
    return measurement_dataframe(parsed, source_file, source_path, provenance)


def show_electrode(parsed, idx=0):
    electrode = parsed["electrodes"][idx]
    fmap = _field_map(electrode.get("fields", []))
    rows = []
    for name, meta in ELECTRODE_FEATURES.items():
        item = fmap.get(name)
        rows.append({
            "Category": meta["category"],
            "Tier": meta["tier"],
            "Field": name,
            "Value": item.get("value") if item else "Not reported",
            "Unit": item.get("unit") if item and item.get("unit") else "—",
            "Conf": item.get("confidence") if item else "—",
            "Source": item.get("source_location") if item else "—",
        })
    df = pd.DataFrame(rows)
    print(
        f"\n[{idx}] {electrode['electrode_id']} | {electrode['label']} | role={electrode['role']}"
        f"{' | PRIMARY' if electrode['is_primary'] else ''}"
    )
    try:
        from IPython.display import display
        display(df.style.hide(axis="index"))
    except Exception:
        print(df.to_string(index=False))

    measurements = measurement_dataframe(
        {"paper": parsed.get("paper", {}), "electrodes": [electrode]},
        provenance="none",
    )
    preview = [
        "measurement_id", "measurement_label", "test_type", "cell_configuration",
        "electrolyte", "current_density", "scan_rate", "gravimetric_capacitance",
        "areal_capacitance", "volumetric_capacitance", "capacitance_retention",
    ]
    preview = [c for c in preview if c in measurements.columns]
    if not measurements.empty:
        print("\nMeasurements:")
        try:
            display(measurements[preview].style.hide(axis="index"))
        except Exception:
            print(measurements[preview].to_string(index=False))
    return df


def show_overview(parsed):
    rows = []
    for electrode in parsed.get("electrodes", []):
        ef = _field_map(electrode.get("fields", []))
        measurements = electrode.get("measurements", [])
        best_cap = None
        best_unit = None
        for measurement in measurements:
            mf = _field_map(measurement.get("fields", []))
            if "gravimetric_capacitance" in mf:
                best_cap = mf["gravimetric_capacitance"]["value"]
                best_unit = mf["gravimetric_capacitance"].get("unit")
                if measurement.get("is_representative"):
                    break
        rows.append({
            "ID": electrode["electrode_id"],
            "Label": electrode["label"],
            "Role": electrode["role"],
            "Formula": ef.get("mxene_formula", {}).get("value", "—"),
            "Synthesis": ef.get("synthesis_method", {}).get("value", "—")[:38],
            "Measurements": len(measurements),
            "Representative Cg": (
                f"{best_cap} {best_unit or ''}".strip() if best_cap is not None else "—"
            ),
            "Electrode fields": len(electrode.get("fields", [])),
        })
    df = pd.DataFrame(rows)
    try:
        from IPython.display import display, Markdown
        display(df.style.hide(axis="index"))
        display(Markdown(f"**Summary**\n\n{parsed['summary']}"))
    except Exception:
        print(df.to_string(index=False))
        print("\nSummary:", parsed.get("summary", ""))
    return df


print("ready")

ready


---

# Step 1 — Point at your PDF

In [8]:
PDF_PATH = "./H2SO4/1-s2.0-S001346862101762X-main.pdf"   # <-- change this

# Colab: uncomment for an upload widget
# from google.colab import files
# up = files.upload(); PDF_PATH = list(up.keys())[0]

assert pathlib.Path(PDF_PATH).exists(), f"not found: {PDF_PATH}"
text, n_pages = extract_pdf_text(PDF_PATH)

pages: 7   chars: 41,652   est. tokens: ~10,413

--- first 400 chars ---
Electrochimica Acta 401 (2022) 139476 
Contents lists available at ScienceDirect 
Electrochimica Acta 
journal homepage: www.elsevier.com/locate/electacta 
High capacitance of MXene (Ti 3 C 2 T x ) through Intercalation and Surface 
Modiﬁcation in Molten Salt 
Liang Guo a , Wei-Yan Jiang b , c , d , Miao Shen b , c , d , ∗, Cong Xu a , Chen-Xu Ding a , 
Su-Fang Zhao b , c , d , Tao-Tao Yuan a , Ch


# Step 2 — Extract

In [9]:
stem = pathlib.Path(PDF_PATH).stem
cache_json = OUTPUT_DIR / f"{stem}_literature_ml.json"

if cache_json.exists():
    rec = json.loads(cache_json.read_text())
    if rec.get("schema_version") != SCHEMA_VERSION:
        raise ValueError(
            f"Cache uses schema {rec.get('schema_version')}; delete {cache_json.name} and rerun."
        )
    parsed = {k: rec[k] for k in ("paper", "electrodes", "n_electrodes", "n_measurements", "summary")}
    usage = {"in": 0, "out": 0, "model": rec.get("model", MODEL)}
    cost = rec.get("cost_usd", 0.0)
    warns = rec.get("warnings", validate(parsed, verbose=False))
    print(f"cached — loaded {cache_json.name} (no API call)")
    print(
        f"found {len(parsed['electrodes'])} electrodes and "
        f"{parsed['n_measurements']} measurements | prior cost ~${cost:.4f}"
    )
    if warns:
        for warning in warns:
            print("  ! " + warning)
else:
    parsed, usage = extract_descriptors(
        text, model=MODEL, measurement_mode=MEASUREMENT_MODE
    )
    print(
        f"\nfound {len(parsed['electrodes'])} electrodes and "
        f"{parsed['n_measurements']} measurements"
    )
    cost = report_cost(
        usage,
        n_elec=len(parsed["electrodes"]),
        n_meas=parsed["n_measurements"],
    )
    print()
    warns = validate(parsed)


found 3 electrodes and 8 measurements
tokens  in=17,324  out=7,252   cost ~$0.1521   [gpt-5.6-terra]   (3 electrodes, 8 measurements)

checks passed


# Step 3 — Review the extracted hierarchy

Start with the electrode overview, then inspect one material and its linked measurements.

In [36]:
_ = show_overview(parsed)

 ID      Label      Role Formula                              Synthesis  Measurements Representative Cg  Electrode fields
E01       CPCM   control       — Hydrothermal chitosan carbon microsphe             2                 —                11
E02 CPCM/MXene composite Ti3C2Tx Electrostatic assembly of HCS with Ti3             8           362 F/g                26

Summary: The paper electrochemically compares two distinct tested electrode materials: the chitosan-derived porous carbon microsphere control (CPCM) and the Ti3C2Tx-containing CPCM/MXene sandwich composite. CPCM/MXene is the primary electrode and was evaluated in three-electrode 1 mol/L H2SO4 and symmetric two-electrode 1 mol/L Na2SO4 configurations. Its explicitly reported headline three-electrode capacitance is 362 F/g at 0.5 A/g, while the symmetric device provides 194.5 F/g at 1 A/g and 27.8 W/(h • kg) at 500.0 W/kg. The detailed Results text reports 93.77% retention after 10,000 cycles at 10 A/g; the abstract instead sta

In [37]:
# Change the index to inspect another electrode and all linked measurements
_ = show_electrode(parsed, 0)


[0] E01 | CPCM | role=control
         Category     Tier                        Field                                                                                       Value  Unit    Conf                            Source
         identity     core                mxene_formula                                                                                Not reported     —       —                                 —
         identity extended             parent_max_phase                                                                                Not reported     —       —                                 —
         identity     core                  composition                                          Chitosan-derived porous carbon microspheres (CPCM)     —  stated    Abstract; Sections 2.3 and 3.1
         identity extended                 mxene_family                                                                                Not reported     —       —                        

# Step 4 — Save

The JSON file is the audit record with evidence and source locations. The two CSVs are linked by `source_file`, `electrode_id`, and `measurement_id`:

- `*_electrodes.csv`: one row per material;
- `*_measurements_ml.csv`: one row per condition-specific result with electrode descriptors included.

The feature catalog is also exported so the meaning and literature basis of every column travel with the dataset.

In [38]:
stem = pathlib.Path(PDF_PATH).stem

electrode_df = electrode_dataframe(
    parsed, source_file=pathlib.Path(PDF_PATH).name, source_path=PDF_PATH, provenance="units"
)
measurement_df = measurement_dataframe(
    parsed, source_file=pathlib.Path(PDF_PATH).name, source_path=PDF_PATH, provenance="units"
)

electrode_df.to_csv(OUTPUT_DIR / f"{stem}_electrodes.csv", index=False)
measurement_df.to_csv(OUTPUT_DIR / f"{stem}_measurements_ml.csv", index=False)
feature_catalog_dataframe().to_csv(OUTPUT_DIR / "mxene_feature_schema.csv", index=False)

with open(OUTPUT_DIR / f"{stem}_literature_ml.json", "w") as fh:
    json.dump(
        {
            "schema_version": SCHEMA_VERSION,
            "measurement_mode": MEASUREMENT_MODE,
            "source_file": PDF_PATH,
            "model": usage["model"],
            "cost_usd": round(cost, 5),
            "warnings": warns,
            **parsed,
        },
        fh,
        indent=2,
    )

print(f"wrote {OUTPUT_DIR}/{stem}_electrodes.csv ({len(electrode_df)} electrode rows)")
print(f"wrote {OUTPUT_DIR}/{stem}_measurements_ml.csv ({len(measurement_df)} measurement rows)")
print(f"wrote {OUTPUT_DIR}/{stem}_literature_ml.json")
print(f"wrote {OUTPUT_DIR}/mxene_feature_schema.csv")
measurement_df.head()

wrote V2/1-s2.0-S2369969821000785-main_electrodes.csv (2 electrode rows)
wrote V2/1-s2.0-S2369969821000785-main_measurements_ml.csv (10 measurement rows)
wrote V2/1-s2.0-S2369969821000785-main_literature_ml.json
wrote V2/mxene_feature_schema.csv


,schema_version,source_file,source_path,authors,doi,journal,title,year,electrode_id,electrode_label,...,relaxation_time__conf,diffusion_coefficient,diffusion_coefficient__unit,diffusion_coefficient__conf,b_value,b_value__unit,b_value__conf,capacitive_contribution,capacitive_contribution__unit,capacitive_contribution__conf
0,2.0-literature-ml,1-s2.0-S2369969821000785-main.pdf,./H2SO4/1-s2.0-S2369969821000785-main.pdf,Lansheng Wei; Weijie Deng; Shanshan Li; Zhengg...,10.1016/j.jobab.2021.10.001,Journal of Bioresources and Bioproducts,Sandwich-like chitosan porous carbon Spheres/M...,2022,E01,CPCM,...,None,None,None,None,None,None,None,None,None,None
1,2.0-literature-ml,1-s2.0-S2369969821000785-main.pdf,./H2SO4/1-s2.0-S2369969821000785-main.pdf,Lansheng Wei; Weijie Deng; Shanshan Li; Zhengg...,10.1016/j.jobab.2021.10.001,Journal of Bioresources and Bioproducts,Sandwich-like chitosan porous carbon Spheres/M...,2022,E01,CPCM,...,None,None,None,None,None,None,None,None,None,None
2,2.0-literature-ml,1-s2.0-S2369969821000785-main.pdf,./H2SO4/1-s2.0-S2369969821000785-main.pdf,Lansheng Wei; Weijie Deng; Shanshan Li; Zhengg...,10.1016/j.jobab.2021.10.001,Journal of Bioresources and Bioproducts,Sandwich-like chitosan porous carbon Spheres/M...,2022,E02,CPCM/MXene,...,None,None,None,None,None,None,None,None,None,None
3,2.0-literature-ml,1-s2.0-S2369969821000785-main.pdf,./H2SO4/1-s2.0-S2369969821000785-main.pdf,Lansheng Wei; Weijie Deng; Shanshan Li; Zhengg...,10.1016/j.jobab.2021.10.001,Journal of Bioresources and Bioproducts,Sandwich-like chitosan porous carbon Spheres/M...,2022,E02,CPCM/MXene,...,None,None,None,None,None,None,None,None,None,None
4,2.0-literature-ml,1-s2.0-S2369969821000785-main.pdf,./H2SO4/1-s2.0-S2369969821000785-main.pdf,Lansheng Wei; Weijie Deng; Shanshan Li; Zhengg...,10.1016/j.jobab.2021.10.001,Journal of Bioresources and Bioproducts,Sandwich-like chitosan porous carbon Spheres/M...,2022,E02,CPCM/MXene,...,None,None,None,None,None,None,None,None,None,None


---

# Batch mode

Each paper is cached as schema-versioned JSON. Old caches from the previous one-row-per-electrode notebook are not reused. The batch produces both an electrode corpus and a measurement-level ML corpus.

Before a full run, hand-check at least 10–20 papers. The most damaging errors are incorrect sample attribution, duplicated values repeated in several sections, and conditions attached to the wrong capacitance. These errors look numerically plausible, so automated schema validation cannot replace manual review.

In [39]:
def estimate_cost(
    folder,
    model=MODEL,
    recursive=True,
    avg_electrodes=3.0,
    avg_measurements_per_electrode=4.0,
):
    """Dry run using real PDF text sizes and an approximate sparse-output budget."""
    folder = pathlib.Path(folder)
    pdfs = sorted(folder.rglob("*.pdf") if recursive else folder.glob("*.pdf"))
    if not pdfs:
        print(f"no PDFs found in {folder}")
        return

    total_in, sizes, unreadable = 0, [], []
    for pdf in pdfs:
        try:
            txt, _ = extract_pdf_text(pdf, verbose=False)
            if len(txt.strip()) < 1000:
                unreadable.append(pdf.name)
            tokens = len(txt) // 4
            sizes.append(tokens)
            total_in += tokens
        except Exception as exc:
            unreadable.append(f"{pdf.name} ({type(exc).__name__})")

    # Sparse records vary widely. These values are intentionally conservative.
    out_per_paper = (
        300
        + 500 * avg_electrodes
        + 260 * avg_electrodes * avg_measurements_per_electrode
    )
    total_out = out_per_paper * len(sizes)
    prompt_overhead = 5000 * len(sizes)  # catalog and extraction instructions
    total_in += prompt_overhead

    print(f"{len(pdfs)} PDFs in {folder}")
    if sizes:
        print(f"input tokens: {total_in:,} (median paper {sorted(sizes)[len(sizes)//2]:,})")
    print(
        f"projected output: ~{int(total_out):,} tokens "
        f"({avg_electrodes} electrodes/paper, "
        f"{avg_measurements_per_electrode} measurements/electrode)"
    )
    if unreadable:
        print(f"\n{len(unreadable)} PDFs with little or no text layer — need OCR:")
        for item in unreadable[:10]:
            print("   " + item)
        if len(unreadable) > 10:
            print(f"   ... and {len(unreadable)-10} more")

    print("\nprojected cost:")
    for model_name, (input_rate, output_rate) in PRICING.items():
        cost = total_in / 1e6 * input_rate + total_out / 1e6 * output_rate
        star = "  <-- current" if model_name == model else ""
        print(f"   {model_name:16s} ${cost:7.2f}   (batch API: ${cost/2:6.2f}){star}")
    return {
        "n_pdfs": len(pdfs),
        "in": total_in,
        "out": int(total_out),
        "unreadable": unreadable,
    }


def batch_extract(
    folder,
    out_dir=OUTPUT_DIR / "extracted_ml_schema_v2",
    model=MODEL,
    sleep=0.5,
    recursive=True,
    limit=None,
    measurement_mode=MEASUREMENT_MODE,
):
    """Extract a PDF folder. Schema-versioned, resumable, and safe to interrupt."""
    folder, out_dir = pathlib.Path(folder), pathlib.Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pdfs = sorted(folder.rglob("*.pdf") if recursive else folder.glob("*.pdf"))
    print(f"{len(pdfs)} PDFs found in {folder}\n")

    results, failures, flagged = [], [], []
    total_cost, total_electrodes, total_measurements, n_done = 0.0, 0, 0, 0

    for index, pdf in enumerate(pdfs, 1):
        rel = pdf.relative_to(folder)
        safe = str(rel.with_suffix("")).replace("/", "__").replace("\\", "__")
        out_json = out_dir / f"{safe}.json"

        if out_json.exists():
            rec = json.loads(out_json.read_text())
            if rec.get("schema_version") == SCHEMA_VERSION and rec.get("measurement_mode") == measurement_mode:
                results.append(rec)
                total_electrodes += len(rec.get("electrodes", []))
                total_measurements += rec.get("n_measurements", 0)
                print(f"[{index}/{len(pdfs)}] {rel} — cached")
                continue
            print(f"[{index}/{len(pdfs)}] {rel} — stale cache, re-extracting", end="  ")
        else:
            print(f"[{index}/{len(pdfs)}] {rel}", end="  ")

        if limit is not None and n_done >= limit:
            print("skipped by limit")
            continue

        try:
            text, _ = extract_pdf_text(pdf, verbose=False)
            if len(text.strip()) < 1000:
                raise ValueError("almost no text layer — likely scanned, needs OCR")

            parsed, usage = extract_descriptors(
                text, model=model, measurement_mode=measurement_mode
            )
            input_rate, output_rate = PRICING.get(model, (2.50, 15.00))
            cost = usage["in"] / 1e6 * input_rate + usage["out"] / 1e6 * output_rate
            total_cost += cost
            n_done += 1

            warnings = validate(parsed, verbose=False)
            rec = {
                "schema_version": SCHEMA_VERSION,
                "measurement_mode": measurement_mode,
                "source_file": pdf.name,
                "source_path": str(rel),
                "model": model,
                "cost_usd": round(cost, 5),
                "warnings": warnings,
                **parsed,
            }
            out_json.write_text(json.dumps(rec, indent=2))
            results.append(rec)

            n_electrodes = len(parsed["electrodes"])
            n_measurements = parsed["n_measurements"]
            total_electrodes += n_electrodes
            total_measurements += n_measurements
            print(
                f"ok — {n_electrodes} electrodes, {n_measurements} measurements, ${cost:.4f}"
                + (f", {len(warnings)} warnings" if warnings else "")
            )
            if warnings:
                flagged.append((pdf.name, warnings))

        except KeyboardInterrupt:
            print("\n\ninterrupted — completed JSON files are saved; rerun to resume")
            break
        except Exception as exc:
            print(f"FAILED — {type(exc).__name__}: {exc}")
            failures.append((pdf.name, str(exc)))
        time.sleep(sleep)

    print("\n" + "=" * 70)
    print(
        f"{len(results)} papers, {total_electrodes} electrodes, "
        f"{total_measurements} measurements"
    )
    print(f"{len(failures)} failed, {len(flagged)} flagged")
    print(f"cost this run: ${total_cost:.2f}")
    return results, failures, flagged


def corpus_dataframes(results, provenance="units"):
    electrode_frames, measurement_frames = [], []
    for record in results:
        if not record.get("electrodes"):
            continue
        kwargs = {
            "source_file": record.get("source_file", ""),
            "source_path": record.get("source_path", ""),
            "provenance": provenance,
        }
        electrode_frames.append(electrode_dataframe(record, **kwargs))
        measurement_frames.append(measurement_dataframe(record, **kwargs))
    electrode_corpus = (
        pd.concat(electrode_frames, ignore_index=True) if electrode_frames else pd.DataFrame()
    )
    measurement_corpus = (
        pd.concat(measurement_frames, ignore_index=True) if measurement_frames else pd.DataFrame()
    )
    return electrode_corpus, measurement_corpus


def corpus_dataframe(results, provenance="units"):
    """Backward-compatible alias returning the measurement-level corpus."""
    return corpus_dataframes(results, provenance=provenance)[1]


print("ready")

ready


In [14]:
# ---- 1. Preflight: estimate corpus size and cost without API calls ----
FOLDER = "./H2SO4"          # <-- your PDF folder

est = estimate_cost(FOLDER)

31 PDFs in H2SO4
input tokens: 735,878 (median paper 14,301)
projected output: ~152,520 tokens (3.0 electrodes/paper, 4.0 measurements/electrode)

projected cost:
   gpt-5.4          $   4.13   (batch API: $  2.06)
   gpt-5.6-terra    $   4.13   (batch API: $  2.06)
   gpt-5.6-sol      $   8.25   (batch API: $  4.13)  <-- current
   gpt-5.6-luna     $   1.65   (batch API: $  0.83)
   gpt-5.4-mini     $   1.24   (batch API: $  0.62)
   gpt-5.4-nano     $   0.34   (batch API: $  0.17)


### 2. Trial run on a few papers

Process five papers first. Compare the enumerated sample list, several capacitance-condition pairs, and at least one cycling or EIS result against the PDFs. The full run reuses compatible caches.

In [15]:
results, failures, flagged = batch_extract(
    FOLDER,
    out_dir=OUTPUT_DIR / "extracted_ml_schema_v2",
    limit=5,
    measurement_mode=MEASUREMENT_MODE,
)

electrode_corpus, measurement_corpus = corpus_dataframes(results)
corpus = measurement_corpus

print(
    f"\n{len(electrode_corpus)} electrodes and {len(measurement_corpus)} measurements "
    f"from {measurement_corpus['source_file'].nunique() if not measurement_corpus.empty else 0} papers"
)
preview_cols = [
    "source_file", "electrode_id", "electrode_label", "mxene_formula",
    "measurement_id", "test_type", "electrolyte", "current_density",
    "scan_rate", "gravimetric_capacitance",
]
preview_cols = [col for col in preview_cols if col in measurement_corpus.columns]
measurement_corpus[preview_cols].head(20)

31 PDFs found in H2SO4

[1/31] 1-s2.0-S2369969821000785-main.pdf  ok — 2 electrodes, 10 measurements, $0.3510
[2/31] 1-s2.0-S2405829723005251-main.pdf  ok — 9 electrodes, 21 measurements, $0.5758, 2 warnings
[3/31] 1399231.pdf  ok — 4 electrodes, 31 measurements, $0.6487, 4 warnings
[4/31] 201807260933092014.pdf  ok — 4 electrodes, 15 measurements, $0.4893, 4 warnings
[5/31] acsnano.9b10066.pdf  ok — 7 electrodes, 17 measurements, $0.4993, 2 warnings
[6/31] Advanced Materials - 2023 - Arslanoglu - 3D Assembly of MXene Networks using a Ceramic Backbone with Controlled Porosity.pdf  skipped by limit
[7/31] Advanced Science - 2023 - He - Wide Temperature All‐Solid‐State Ti3C2Tx Quantum Dots L‐Ti3C2Tx Fiber Supercapacitor with.pdf  skipped by limit
[8/31] ChemElectroChem - 2021 - Abdolhosseinzadeh - Coating Porous MXene Films with Tunable Porosity for High‐Performance.pdf  skipped by limit
[9/31] d5ta06789h.pdf  skipped by limit
[10/31] d6ta00010j.pdf  skipped by limit
[11/31] em4045_down.

,source_file,electrode_id,electrode_label,mxene_formula,measurement_id,test_type,electrolyte,current_density,scan_rate,gravimetric_capacitance
0,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,Ti3C2Tx,E01-M01,GCD,H2SO4,0.5,None,362
1,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,Ti3C2Tx,E01-M02,rate capability from CV,1 mol/L H2SO4,NaN,None,NaN
2,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,Ti3C2Tx,E01-M03,GCD,1 mol/L H2SO4,NaN,None,NaN
3,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,Ti3C2Tx,E01-M04,EIS equivalent-circuit fitting,1 mol/L H2SO4,NaN,None,NaN
4,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,Ti3C2Tx,E01-M05,cycling,1 mol/L H2SO4,10,None,NaN
5,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,Ti3C2Tx,E01-M06,GCD,Na2SO4,1,None,194.5
6,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,Ti3C2Tx,E01-M07,GCD-derived Ragone performance,1 mol/L Na2SO4,NaN,None,NaN
7,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,Ti3C2Tx,E01-M08,GCD-derived Ragone performance,1 mol/L Na2SO4,NaN,None,NaN
8,1-s2.0-S2369969821000785-main.pdf,E02,CPCM,NaN,E02-M01,GCD,1 mol/L H2SO4,NaN,None,NaN
9,1-s2.0-S2369969821000785-main.pdf,E02,CPCM,NaN,E02-M02,EIS equivalent-circuit fitting,1 mol/L H2SO4,NaN,None,NaN


### 3. Full run

Remove `limit` to process the full folder. Compatible JSON files are reused, and each newly completed paper is saved immediately.

In [40]:
results, failures, flagged = batch_extract(
    FOLDER,
    out_dir=OUTPUT_DIR / "extracted_ml_schema_v2",
    measurement_mode=MEASUREMENT_MODE,
)

electrode_corpus, measurement_corpus = corpus_dataframes(results)
corpus = measurement_corpus

electrode_corpus.to_csv(OUTPUT_DIR / "h2so4_electrodes.csv", index=False)
measurement_corpus.to_csv(OUTPUT_DIR / "h2so4_measurements_ml.csv", index=False)
feature_catalog_dataframe().to_csv(OUTPUT_DIR / "mxene_feature_schema.csv", index=False)

print(
    f"\nwrote {OUTPUT_DIR}/h2so4_electrodes.csv — {len(electrode_corpus)} electrode rows\n"
    f"wrote {OUTPUT_DIR}/h2so4_measurements_ml.csv — {len(measurement_corpus)} measurement rows\n"
    f"wrote {OUTPUT_DIR}/mxene_feature_schema.csv — {len(feature_catalog_dataframe())} feature definitions"
)

31 PDFs found in H2SO4

[1/31] 1-s2.0-S2369969821000785-main.pdf — cached
[2/31] 1-s2.0-S2405829723005251-main.pdf — cached
[3/31] 1399231.pdf — cached
[4/31] 201807260933092014.pdf — cached
[5/31] acsnano.9b10066.pdf — cached
[6/31] Advanced Materials - 2023 - Arslanoglu - 3D Assembly of MXene Networks using a Ceramic Backbone with Controlled Porosity.pdf  ok — 6 electrodes, 17 measurements, $0.1998, 13 warnings
[7/31] Advanced Science - 2023 - He - Wide Temperature All‐Solid‐State Ti3C2Tx Quantum Dots L‐Ti3C2Tx Fiber Supercapacitor with.pdf  ok — 8 electrodes, 20 measurements, $0.1826
[8/31] ChemElectroChem - 2021 - Abdolhosseinzadeh - Coating Porous MXene Films with Tunable Porosity for High‐Performance.pdf  ok — 4 electrodes, 7 measurements, $0.1145, 5 warnings
[9/31] d5ta06789h.pdf  ok — 3 electrodes, 26 measurements, $0.1995, 1 warnings
[10/31] d6ta00010j.pdf  ok — 15 electrodes, 17 measurements, $0.1851, 15 warnings
[11/31] em4045_down.pdf  ok — 8 electrodes, 28 measurements, $0

### 3b. Optional rule-based corpus filter: Ti3C2 + LiF/HCl + H2SO4

The main corpus is measurement-level, so each row has the parent electrode descriptors plus its own electrolyte and condition. The deterministic filter labels each measurement row as PASS, FAIL, or AMBIGUOUS:

- primary formula contains Ti3C2;
- synthesis mentions both LiF and HCl;
- the linked measurement uses H2SO4/sulfuric acid.

Retaining the reasoning columns makes each decision auditable.

In [41]:
import unicodedata

_SUBSCRIPT_MAP = str.maketrans("₀₁₂₃₄₅₆₇₈₉ₓ", "0123456789x")
_NULL_PATTERN = re.compile(
    r"not\s+reported|not\s+available|not\s+specified|not\s+applicable"
    r"|\bn/a\b|\bna\b|\bnone\b|\bunknown\b|\bunspecified\b|^nan$",
    re.IGNORECASE,
)


def _normalise(text: str) -> str:
    return unicodedata.normalize("NFKC", text).translate(_SUBSCRIPT_MAP).lower()


def _clean_text(value):
    if value is None or pd.isna(value):
        return ""
    text = str(value).strip()
    return "" if _NULL_PATTERN.search(text) else text


def check_formula(formula: str):
    norm = _normalise(formula)
    primary = re.split(r";|/|,|\(secondary\)|\(minor\)|\(second", norm)[0].strip()
    if re.search(r"ti3c2", primary):
        return True, f"Primary formula '{formula}' contains Ti3C2."
    if re.search(r"ti3c2", norm):
        return False, f"Ti3C2 appears in '{formula}' but is not the primary formula."
    if not norm:
        return None, "Formula is empty."
    return False, f"Primary formula '{formula}' is not Ti3C2."


def check_electrolyte(electrolyte: str):
    if not electrolyte:
        return None, "Electrolyte is empty."
    norm = _normalise(electrolyte)
    if re.search(r"h2so4|su[lp]phuric[\s\-]?acid", norm):
        return True, "Electrolyte explicitly mentions H2SO4."
    return False, f"Electrolyte '{electrolyte}' is not H2SO4."


def check_synthesis(synthesis: str):
    if not synthesis:
        return None, "Synthesis is empty."
    norm = _normalise(synthesis)
    has_lif = bool(re.search(r"\blif\b|lithium[\s\-]?fluoride", norm))
    has_hcl = bool(re.search(r"\bhcl\b|hydrochloric[\s\-]?acid", norm))
    if has_lif and has_hcl:
        return True, "Synthesis explicitly mentions both LiF and HCl."
    if not has_lif and not has_hcl:
        return False, "Synthesis mentions neither LiF nor HCl."
    return False, "Synthesis mentions only one of LiF or HCl."


def evaluate_row(formula: str, synthesis: str, electrolyte: str) -> dict:
    formula_match, formula_reason = check_formula(formula)
    synthesis_match, synthesis_reason = check_synthesis(synthesis)
    electrolyte_match, electrolyte_reason = check_electrolyte(electrolyte)

    checks = (formula_match, synthesis_match, electrolyte_match)
    if any(value is None for value in checks):
        overall = "AMBIGUOUS"
    elif all(checks):
        overall = "PASS"
    else:
        overall = "FAIL"

    return {
        "formula_match": formula_match,
        "formula_reasoning": formula_reason,
        "synthesis_match": synthesis_match,
        "synthesis_reasoning": synthesis_reason,
        "electrolyte_match": electrolyte_match,
        "electrolyte_reasoning": electrolyte_reason,
        "overall": overall,
    }


def filter_corpus(df):
    verdicts = []
    for _, row in df.iterrows():
        verdicts.append(
            evaluate_row(
                _clean_text(row.get("mxene_formula")),
                _clean_text(row.get("synthesis_method")),
                _clean_text(row.get("electrolyte")),
            )
        )
    return pd.concat([df, pd.DataFrame(verdicts, index=df.index)], axis=1)


corpus_filtered = filter_corpus(corpus)
passed = corpus_filtered[corpus_filtered["overall"] == "PASS"]
failed = corpus_filtered[corpus_filtered["overall"] == "FAIL"]
ambiguous = corpus_filtered[corpus_filtered["overall"] == "AMBIGUOUS"]

print(f"{len(corpus_filtered)} measurement rows filtered")
print(f"  PASS      : {len(passed):>4}")
print(f"  FAIL      : {len(failed):>4}")
print(f"  AMBIGUOUS : {len(ambiguous):>4}")

corpus_filtered.to_csv(OUTPUT_DIR / "h2so4_measurements_filtered.csv", index=False)
print(f"\nwrote {OUTPUT_DIR}/h2so4_measurements_filtered.csv")

show_cols = [
    "source_file", "electrode_id", "electrode_label", "measurement_id",
    "mxene_formula", "synthesis_method", "electrolyte", "gravimetric_capacitance",
]
show_cols = [col for col in show_cols if col in passed.columns]
passed[show_cols].head(20)

498 measurement rows filtered
  PASS      :   82
  FAIL      :   49
  AMBIGUOUS :  367

wrote V2/h2so4_measurements_filtered.csv


,source_file,electrode_id,electrode_label,measurement_id,mxene_formula,synthesis_method,electrolyte,gravimetric_capacitance
0,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,E01-M01,Ti3C2Tx,"hydrothermal synthesis, LiF/HCl MXene etching,...",H2SO4,362
1,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,E01-M02,Ti3C2Tx,"hydrothermal synthesis, LiF/HCl MXene etching,...",1 mol/L H2SO4,NaN
2,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,E01-M03,Ti3C2Tx,"hydrothermal synthesis, LiF/HCl MXene etching,...",1 mol/L H2SO4,NaN
3,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,E01-M04,Ti3C2Tx,"hydrothermal synthesis, LiF/HCl MXene etching,...",1 mol/L H2SO4,NaN
4,1-s2.0-S2369969821000785-main.pdf,E01,CPCM/MXene,E01-M05,Ti3C2Tx,"hydrothermal synthesis, LiF/HCl MXene etching,...",1 mol/L H2SO4,NaN
38,1399231.pdf,E02,M/G-1%,E02-M01,Ti3C2Tx,hydrazine-reduced GO and LiF/HCl-etched MXene ...,3 M H2SO4,308.0
39,1399231.pdf,E02,M/G-1%,E02-M02,Ti3C2Tx,hydrazine-reduced GO and LiF/HCl-etched MXene ...,3 M H2SO4,NaN
40,1399231.pdf,E02,M/G-1%,E02-M03,Ti3C2Tx,hydrazine-reduced GO and LiF/HCl-etched MXene ...,3 M H2SO4,NaN
41,1399231.pdf,E03,M/G-5%,E03-M01,Ti3C2Tx,hydrazine-reduced GO and LiF/HCl-etched MXene ...,H2SO4 aqueous electrolyte,335.4
42,1399231.pdf,E03,M/G-5%,E03-M02,Ti3C2Tx,hydrazine-reduced GO and LiF/HCl-etched MXene ...,3 M H2SO4,NaN


In [42]:
passed.to_csv(OUTPUT_DIR / "h2so4_measurements_passed.csv", index=False)
print(f"wrote {OUTPUT_DIR}/h2so4_measurements_passed.csv (PASS measurement rows only)")

wrote V2/h2so4_measurements_passed.csv (PASS measurement rows only)


### 4. Corpus coverage and data quality

Coverage is reported separately for core and extended features. A feature can be scientifically important but still unusable as a standalone ML input when most papers do not report it. The schema remains broad so sparse descriptors can later be used for focused subsets, missingness analysis, or targeted manual completion.

In [43]:
# Measurements and electrodes per paper — inspect extreme contributors.
if corpus.empty:
    print("corpus is empty")
else:
    measurements_per_paper = corpus.groupby("source_file").size().sort_values(ascending=False)
    electrodes_per_paper = (
        corpus[["source_file", "electrode_id"]]
        .drop_duplicates()
        .groupby("source_file")
        .size()
        .sort_values(ascending=False)
    )
    print("electrodes per paper:")
    print(
        f"  median {electrodes_per_paper.median():.0f} | "
        f"max {electrodes_per_paper.max()} | min {electrodes_per_paper.min()}"
    )
    print("measurements per paper:")
    print(
        f"  median {measurements_per_paper.median():.0f} | "
        f"max {measurements_per_paper.max()} | min {measurements_per_paper.min()}"
    )
    if (measurements_per_paper > 100).any():
        print("\nPapers with >100 measurements — check for duplicate narrative values:")
        print(measurements_per_paper[measurements_per_paper > 100].head(10).to_string())

electrodes per paper:
  median 4 | max 10 | min 1
measurements per paper:
  median 17 | max 31 | min 7


In [44]:
catalog = feature_catalog_dataframe().set_index("feature")
feature_cols = [
    name for name in list(ELECTRODE_FEATURES) + list(MEASUREMENT_FEATURES)
    if name in corpus.columns
]

coverage_rows = []
for name in feature_cols:
    meta = catalog.loc[name]
    filled = corpus[name].notna().sum()
    pct = round(100 * corpus[name].notna().mean(), 1) if len(corpus) else 0.0
    coverage_rows.append({
        "level": meta["level"],
        "category": meta["category"],
        "tier": meta["tier"],
        "feature": name,
        "filled": filled,
        "pct": pct,
        "model_status": "usable" if pct >= 60 else ("marginal" if pct >= 30 else "sparse"),
    })

coverage = pd.DataFrame(coverage_rows).sort_values(
    ["tier", "pct"], ascending=[True, False]
)
print(f"coverage across {len(corpus)} measurement rows:\n")
coverage.head(100)

coverage across 498 measurement rows:



,level,category,tier,feature,filled,pct,model_status
2,electrode,identity,core,composition,440,88.4,usable
12,electrode,identity,core,electrode_architecture,401,80.5,usable
68,measurement,test protocol,core,test_type,400,80.3,usable
0,electrode,identity,core,mxene_formula,348,69.9,usable
13,electrode,identity,core,morphology,264,53.0,marginal
...,...,...,...,...,...,...,...
97,measurement,target,extended,specific_capacity,3,0.6,sparse
75,measurement,electrolyte,extended,solvent,1,0.2,sparse
39,electrode,structure,extended,micropore_volume,0,0.0,sparse
40,electrode,structure,extended,mesopore_volume,0,0.0,sparse


In [45]:
print("stated / derived / uncertain by feature:\n")
for feature in feature_cols:
    conf_col = feature + "__conf"
    if conf_col not in corpus.columns:
        continue
    counts = corpus[conf_col].value_counts()
    total = counts.sum()
    if total == 0:
        continue
    stated = counts.get("stated", 0)
    derived = counts.get("derived", 0)
    uncertain = counts.get("uncertain", 0)
    warning = "  <-- mostly inherited" if derived > stated else ""
    print(f"{feature:32s} {stated:5d} / {derived:5d} / {uncertain:5d}{warning}")

stated / derived / uncertain by feature:

mxene_formula                      314 /    34 /     0
parent_max_phase                   238 /    57 /     0
composition                        440 /     0 /     0
mxene_family                        33 /    56 /     0  <-- mostly inherited
composite_partner                   93 /     1 /     0
composite_ratio                    163 /     7 /     0
mxene_weight_fraction               77 /    10 /     0
dopant_elements                     17 /     0 /     0
dopant_content                       7 /     0 /     0
intercalant                         71 /     4 /    18
preintercalated_ion                 34 /     0 /    15
surface_termination_summary        168 /     0 /     0
electrode_architecture             354 /    47 /     0
morphology                         258 /     6 /     0
layer_state                        118 /    21 /     0
synthesis_method                   187 /    36 /     0
etchant                            170 /    55 /     0
e

### 5. Papers requiring manual attention

Failures are often scanned PDFs without a usable text layer. Flagged papers completed extraction but violated one or more consistency checks.

In [46]:
if failures:
    print(f"{len(failures)} failed:\n")
    for name, err in failures:
        print(f"   {name}\n      {err}")
    print("\nFor scanned PDFs:  ocrmypdf in.pdf out.pdf   then rerun batch_extract")
else:
    print("no failures")

if flagged:
    print(f"\n{len(flagged)} flagged:\n")
    for name, ws in flagged:
        print(f"   {name}")
        for w in ws:
            print(f"      ! {w}")
else:
    print("\nnothing flagged")

no failures

24 flagged:

   Advanced Materials - 2023 - Arslanoglu - 3D Assembly of MXene Networks using a Ceramic Backbone with Controlled Porosity.pdf
      ! 'MX-PS supercapacitor, 60% longitudinal porosity, 180 mg mL−1 MXene infiltration' duplicate electrode fields: ['mass_loading']
      ! 'E01-M01' has no target or related electrochemical output
      ! 'E01-M02' has no target or related electrochemical output
      ! 'E01-M04' has no target or related electrochemical output
      ! 'E01-M05' has no target or related electrochemical output
      ! 'E01-M06' has no target or related electrochemical output
      ! 'E01-M07' has no target or related electrochemical output
      ! 'E03-M01' has no target or related electrochemical output
      ! 'E04-M01' has no target or related electrochemical output
      ! 'E05-M01' has no target or related electrochemical output
      ! 'E05-M02' has no target or related electrochemical output
      ! 'E06-M01' has no target or related electroc

### 6. Inspect one paper against the PDF

Review the sample enumeration first, then inspect whether the measurement conditions and targets are correctly linked. Manual validation should include papers with many variants, multiple cell configurations, and values reported mainly in tables or captions.

In [47]:
import random

if corpus.empty:
    print("corpus is empty")
else:
    check = random.choice(corpus["source_file"].dropna().unique())
    # check = "your_paper.pdf"

    subset = corpus[corpus["source_file"] == check]
    print(
        f"{check} — {subset['electrode_id'].nunique()} electrodes, "
        f"{len(subset)} measurements\n"
    )
    inspect_cols = [
        "electrode_id", "electrode_label", "role", "mxene_formula",
        "measurement_id", "test_type", "cell_configuration", "electrolyte",
        "current_density", "scan_rate", "potential_window",
        "gravimetric_capacitance", "areal_capacitance", "volumetric_capacitance",
        "capacitance_retention", "esr", "charge_transfer_resistance",
    ]
    inspect_cols = [col for col in inspect_cols if col in subset.columns]
    print(subset[inspect_cols].to_string(index=False))

    record = next(r for r in results if r.get("source_file") == check)
    print("\nmodel summary:\n  " + record["summary"])
    print("\nwarnings:")
    for warning in record.get("warnings", []):
        print("  ! " + warning)

Reassembly-of-MXene-Hydrogels-into-Flexible-Films-towards-Compact-and-Ultrafast-Supercapacitors.pdf — 3 electrodes, 14 measurements

electrode_id electrode_label    role mxene_formula measurement_id         test_type cell_configuration               electrolyte current_density scan_rate potential_window gravimetric_capacitance areal_capacitance volumetric_capacitance capacitance_retention  esr charge_transfer_resistance
         E01    Ti3C2Tx film control       Ti3C2Tx        E01-M01               NaN    three-electrode H2SO4 aqueous electrolyte             NaN       NaN             None                    None              None                    NaN                   NaN None                       None
         E01    Ti3C2Tx film control       Ti3C2Tx        E01-M02               NaN                NaN                       NaN             NaN       NaN             None                    None              None                    NaN                   NaN None                      

---

## Modeling rules for the extracted corpus

**Use the measurement table as the modeling table.** The target belongs to a specific electrolyte, cell configuration, potential window, and current density or scan rate. The electrode table is for sample-level deduplication, coverage checks, and materials analysis.

**Normalize units after extraction.** Raw values and units remain separate so conversions are explicit and auditable. Do not combine gravimetric, areal, and volumetric capacitance into one target column.

**Split by paper, not by row.** Multiple measurements from one paper share the same samples, procedures, laboratory, and often repeated sweeps. Random row splitting leaks this information. Use `GroupKFold` or `GroupShuffleSplit` with `source_file` or DOI as the group. For stricter tests, group by both paper and `electrode_id`.

```python
from sklearn.model_selection import GroupKFold

groups = corpus["source_file"]
cv = GroupKFold(n_splits=5)
```

**Deduplicate repeated results.** In `all_explicit` mode, check duplicates across electrode, test type, condition, target value, and unit. Values repeated in the abstract and conclusion should not become separate rows.

**Treat missingness as information.** Begin with core features and report coverage. Extended features can support focused subsets, but imputing a descriptor reported in only a small fraction of papers can create more bias than signal.

**Retain provenance during auditing.** Run `corpus_dataframes(results, provenance="full")` when inspecting extraction quality. The full JSON always retains source locations and short evidence excerpts.

**Do not interpret high random-split accuracy as generalization.** Literature-derived measurements are clustered by paper and electrode. Grouped validation is mandatory before claiming prediction performance.